In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
class underwaterSegmentationDataset(Dataset):
  def __init__(self, image_paths, mask_paths, transform=None, target_transform=None):
    self.image_paths = image_paths
    image_paths.sort()
    self.mask_paths = mask_paths
    mask_paths.sort()
    self.transform = transform
    self.target_transform = target_transform

  def __len__(self):
    return len(self.image_paths)

  def __getitem__(self, idx):

    image = Image.open(self.image_paths[idx]).convert("RGB")
    mask = Image.open(self.mask_paths[idx])

    # Apply transforms
    if self.transform:
      image = self.transform(image)

    if self.target_transform:
      mask = self.target_transform(mask)
      mask = remap_mask(mask)

    return image, mask

In [ ]:
import glob

image_path = os.path.join(path, "dataset", "images", "*.jpg")
image_paths = glob.glob(image_path)

mask_path = os.path.join(path, "dataset", "masks", "*.png")
mask_paths = glob.glob(mask_path)


In [ ]:
train_images, test_images, train_masks, test_masks = train_test_split(
  image_paths, mask_paths, test_size=0.2, random_state=42
)

In [ ]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

# Define transforms for images and masks
img_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

target_transforms = transforms.Compose([                    ## Notice how we have a transform for the target (because it is an image)
    transforms.Resize((256, 256)),                          ##        and another one for the image itself.
    transforms.PILToTensor()                                           ## Question: What do you think would happen if we
                                                            ##  added rotation augmentation to the image only?
])

In [ ]:
train_dataset = underwaterSegmentationDataset(train_images, train_masks, transform=img_transforms, target_transform=target_transforms)
test_dataset = underwaterSegmentationDataset(test_images, test_masks, transform=img_transforms, target_transform=target_transforms)

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
import matplotlib.pyplot as plt

# Display some images with their masks
for i in range(7,10):
    img, mask = train_dataset[i]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img.permute(1, 2, 0))  # Convert (C, H, W) to (H, W, C)
    axes[0].set_title("Image")
    axes[0].axis("off")
    axes[1].imshow(mask.squeeze())
    axes[1].set_title("Segmentation Mask")
    axes[1].axis("off")
    plt.show()

In [ ]:
# TO DO
import segmentation_models_pytorch as smp

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
model = smp.Unet(
  encoder_name="efficientnet-b1",
  encoder_weights="imagenet",
  in_channels=3,
  classes=8,
).to(device)

model = model.to(device)

In [ ]:
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

# Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
        images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long)  # mask shape becomes [N, H, W]

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

# Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long)    # mask shape becomes [N, H, W]

            outputs = model(images)  # Now [N, H, W]
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
# TO DO
import torch.nn as nn

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

num_epochs = 10  # Define number of epochs
train_losses = []
val_losses = []

# Training Loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

In [ ]:
# Plot Training Loss Curve**
import matplotlib.pyplot as plt

plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()

In [ ]:
# TO DO
import random
import matplotlib.pyplot as plt

model.eval()
# Get some test samples
test_samples = random.sample(range(len(test_dataset)), 5)

for idx in test_samples:
    img, mask = test_dataset[idx]

    with torch.no_grad():
        pred_mask = model(img.unsqueeze(0).to(device))  # Forward pass

    pred_mask = torch.softmax(pred_mask, dim=1)  # Convert logits to probabilities
    pred_mask = pred_mask.argmax(dim=1).cpu().squeeze().numpy()  # Get class with highest probability

    # Display images
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img.permute(1, 2, 0))
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(mask.squeeze())
    axes[1].set_title("Ground Truth Mask")
    axes[1].axis("off")

    axes[2].imshow(pred_mask)  # Show class map
    axes[2].set_title("Predicted Mask")
    axes[2].axis("off")

    plt.show()
